In [4]:
import pandas as pd
import numpy as np
from math import cos, sin

## Forward Kinematics

$$
^0_1T = \begin{bmatrix}
C_1 & -S_1 & 0 & 0 \\
S_1 & C_1 & 0 & 0\\
0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1 \\
\end{bmatrix}
$$

$$
^1_2T = \begin{bmatrix}
C_2 & 0 & S_2 & 1.3 \\
0 & 1 & 0 & 40\\
-S_2 & 0 & C_2 & 95 \\
0 & 0 & 0 & 1 \\
\end{bmatrix}
$$

---

$$
^0_2T = \begin{bmatrix}
C_1C_2 & -S_1 & C_1S_2 & 1.3C_1-40S_1 \\
S_1C_2 & C_1 & S_1S_2 & 40C_1+1.3S_1\\
-S_2 & 0 & C_2 & 95 \\
0 & 0 & 0 & 1 \\
\end{bmatrix}
$$

$$
^2_3T = \begin{bmatrix}
C_3 & 0 & S_3 & -133.3 \\
0 & 1 & 0 & -27.5\\
-S_3 & 0 & C_3 & 0.5 \\
0 & 0 & 0 & 1 \\
\end{bmatrix}
$$

---

$$
^0_3T = \begin{bmatrix}
C_1C_{23} & -S_1 & C_1S_{23} & -133.3C_1C_2+0.5C_1S_2+1.3C_1-12.5S_1 \\
S_1C_{23} & C_1 & S_1S_{23} & -133.3S_1C_2+0.5S_1S_2+1.3S_1+12.5C_1\\
-S_{23} & 0 & C_{23} & 0.5C_2+133.3S_2+95 \\
0 & 0 & 0 & 1 \\
\end{bmatrix}
$$

$$
^3_{ee}T = \begin{bmatrix}
1 & 0 & 0 & -126.994 \\
0 & 1 & 0 & -12.2355 \\
0 & 0 & 1 & 2.8614 \\
0 & 0 & 0 & 1 \\
\end{bmatrix}
$$

---

$$
^0_{ee}T = \begin{bmatrix}
C_1C_{23} & -S_1 & C_1S_{23} & 2.8614C_1S_{23}-126.994C_1C_{23}-133.3C_1C_2+0.5C_1S_2+1.3C_1-0.2645S_1 \\
S_1C_{23} & C_1 & S_1S_{23} & 2.8614S_1S_{23}-126.994S_1C_{23}-133.3S_1C_2+0.5S_1S_2+1.3S_1+0.2645C_1 \\
-S_{23} & 0 & C_{23} & 126.994S_{23}+2.8614C_{23}+0.5C_2+133.3S_2+95 \\
0 & 0 & 0 & 1 \\
\end{bmatrix}
$$

> **Solutions**:  
> $$
> ^0_{ee}T = \begin{bmatrix}
> C_1C_{23} & -S_1 & C_1S_{23} & C_1(2.8614S_{23}-126.994C_{23}-133.3C_2+0.5S_2+1.3)-0.2645S_1 \\
> S_1C_{23} & C_1 & S_1S_{23} & S_1(2.8614S_{23}-126.994C_{23}-133.3C_2+0.5S_2+1.3)+0.2645C_1 \\
> -S_{23} & 0 & C_{23} & 126.994S_{23}+2.8614C_{23}+0.5C_2+133.3S_2+95 \\
> 0 & 0 & 0 & 1 \\
> \end{bmatrix}
> $$

In [ ]:
def create_sample_data(theta1, theta2, theta3):
    s1 = sin(theta1); s2 = sin(theta2)
    c1 = cos(theta1); c2 = cos(theta2)
    s23 = sin(theta2 + theta3); c23 = cos(theta2 + theta3)
    chunk3 = -133.3*c2+0.5*s2+1.3
    chunkE = 2.8614*s23-126.994*c23-133.3*c2+0.5*s2+1.3

    joint1 = [[c1, -s1, 0, 0],
              [s1, c1, 0, 0],
              [0, 0, 1, 0],
              [0, 0, 0, 1]]
    joint2 = [[c1*c2, -s1, c1*s2, 1.3*c1-40*s1],
              [s1*c2, c1, s1*s2, -1.3*s1+40*c1],
              [-s2, 0, c2, 95],
              [0, 0, 0, 1]]
    joint3 = [[c1*c23, -s1, c1*s23, c1*chunk3-12.5*s1],
              [s1*c23, c1, s1*s23, s1*chunk3+12.5*c1],
              [-s23, 0, c23, 0.5*c2+133.3*s2+95],
              [0, 0, 0, 1]]
    jointE = [[c1*c23, -s1, c1*s23, c1*chunkE-0.2645*s1],
              [s1*c23, c1, s1*s23, s1*chunkE+0.2645*c1],
              [-s23, 0, c23, 0.5*c2+133.3*s2+95],
              [0, 0, 0, 1]]
    return joint1, joint2, joint3, jointE

def create_dataset(theta1: np.ndarray, theta2: np.ndarray, theta3: np.ndarray):
    """
    Create a dataset of joint transformation matrices for a set of joint angles.
    Each row stores the 4x4 matrices for joint1, joint2, joint3, and jointE.
    """

    # Precompute trig terms
    c1 = np.cos(theta1); s1 = np.sin(theta1)
    c2 = np.cos(theta2); s2 = np.sin(theta2)
    c23 = np.cos(theta2 + theta3); s23 = np.sin(theta2 + theta3)

    chunk3 = -133.3 * c2 + 0.5 * s2 + 1.3
    chunkE = 2.8614 * s23 - 126.994 * c23 - 133.3 * c2 + 0.5 * s2 + 1.3

    # Number of configurations
    N = len(theta1)

    # Allocate transformation arrays (N, 4, 4)
    joint1 = np.zeros((N, 4, 4))
    joint2 = np.zeros((N, 4, 4))
    joint3 = np.zeros((N, 4, 4))
    jointE = np.zeros((N, 4, 4))

    # ---- Fill joint1 ----
    joint1[:,0,0] = c1
    joint1[:,0,1] = -s1
    joint1[:,1,0] = s1
    joint1[:,1,1] = c1
    joint1[:,2,2] = 1
    joint1[:,3,3] = 1

    # ---- Fill joint2 ----
    joint2[:,0,0] = c1 * c2
    joint2[:,0,1] = -s1
    joint2[:,0,2] = c1 * s2
    joint2[:,0,3] = 1.3 * c1 - 40 * s1

    joint2[:,1,0] = s1 * c2
    joint2[:,1,1] = c1
    joint2[:,1,2] = s1 * s2
    joint2[:,1,3] = -1.3 * s1 + 40 * c1

    joint2[:,2,0] = -s2
    joint2[:,2,2] = c2
    joint2[:,2,3] = 95

    joint2[:,3,3] = 1

    # ---- Fill joint3 ----
    joint3[:,0,0] = c1 * c23
    joint3[:,0,1] = -s1
    joint3[:,0,2] = c1 * s23
    joint3[:,0,3] = c1 * chunk3 - 12.5 * s1

    joint3[:,1,0] = s1 * c23
    joint3[:,1,1] = c1
    joint3[:,1,2] = s1 * s23
    joint3[:,1,3] = s1 * chunk3 + 12.5 * c1

    joint3[:,2,0] = -s23
    joint3[:,2,2] = c23
    joint3[:,2,3] = 0.5 * c2 + 133.3 * s2 + 95

    joint3[:,3,3] = 1

    # ---- Fill jointE ----
    jointE[:,0,0] = c1 * c23
    jointE[:,0,1] = -s1
    jointE[:,0,2] = c1 * s23
    jointE[:,0,3] = c1 * chunkE - 0.2645 * s1

    jointE[:,1,0] = s1 * c23
    jointE[:,1,1] = c1
    jointE[:,1,2] = s1 * s23
    jointE[:,1,3] = s1 * chunkE + 0.2645 * c1

    jointE[:,2,0] = -s23
    jointE[:,2,2] = c23
    jointE[:,2,3] = 0.5 * c2 + 133.3 * s2 + 95

    jointE[:,3,3] = 1

    # ---- Create DataFrame ----
    df = pd.DataFrame({
        'theta1': theta1,
        'theta2': theta2,
        'theta3': theta3,
        'joint1': list(joint1),
        'joint2': list(joint2),
        'joint3': list(joint3),
        'jointE': list(jointE)
    })

    return df
    

In [6]:
# Meshgrid from -pi to pi for each theta
theta1 = np.linspace(-np.pi, np.pi, 10)
theta2 = np.linspace(-np.pi, np.pi, 10)
theta3 = np.linspace(-np.pi, np.pi, 10)
theta1, theta2, theta3 = np.meshgrid(theta1, theta2, theta3)
theta1 = theta1.flatten()
theta2 = theta2.flatten()
theta3 = theta3.flatten()
df = create_dataset(theta1, theta2, theta3)
df.to_csv('robotic_arm_data.csv', index=False)